# KC1 extra members (teammate GPU)

Trains two all-data members of our KC1 knee-centred model (ConvNeXt-S, KC256), one on each T4, for team rsna-knee.
Inputs are private team datasets (RSNA competition data only, no external data). Stops itself at 9.0 h.
Run with: Accelerator GPU T4 x2, Internet ON, then Save Version -> Save & Run All (Commit). Do not edit.

In [ ]:
# 1. Inputs: link the 4 store parts into one store dir, copy the code, check the code md5s and the RSNA-only rules.
import os, glob, json, hashlib, re, csv, shutil, subprocess, sys, time
T0 = time.time()
R = '/kaggle/working/rsna'
os.makedirs(R, exist_ok=True)
def find_dir(name):
    c = [p for p in glob.glob('/kaggle/input/**/' + name, recursive=True) if os.path.isdir(p)]
    assert c, 'missing input dir ' + name
    return c[0]
st = os.path.join(R, 'kc_store', 'store'); os.makedirs(st, exist_ok=True)
n = 0
for k in range(4):
    part = find_dir('kc_store_part%d' % k)
    for p in glob.glob(os.path.join(part, '**', '*.npz'), recursive=True):
        d = os.path.join(st, os.path.basename(p))
        if not os.path.exists(d):
            os.symlink(p, d); n += 1
print('store npz linked', n, flush=True)
bundle = os.path.dirname(find_dir('code2'))            # .../fast_build
broot = os.path.dirname(bundle)
shutil.copytree(os.path.join(broot, 'fast_build'), os.path.join(R, 'fast_build'), dirs_exist_ok=True)
for sub in ('r7port', 'hf', 'kc', 'r7t_labels', 'rsna_train'):
    src = find_dir(sub) if not os.path.isdir(os.path.join(broot, sub)) else os.path.join(broot, sub)
    if not os.path.exists(os.path.join(R, sub)):
        os.symlink(src, os.path.join(R, sub))
CODE_MD5 = {"aug_dump.py": "d6498cd6ba093f9f5c228705ee1c4316", "fleet_aug.py": "4e7311c2f4c6f94acfb1a1abe1bfc12f", "fleet_common.py": "658ab8103acdf6e4aec029a0f6ee5959", "fleet_model.py": "897f83b272fbb144a4d3deab773d9064", "fleet_model_f2.py": "0122f791c82130d5df292dada522eac1", "fleet_model_ttr.py": "bcdf7d153f339962b97d2736d8baed08", "fleet_train.py": "8d12515f237d204ad63ca93bd74b36b9", "fleet_train_kc.py": "f1d266cfc424f9913fbdd23c11e6dd9c", "fleet_worker.py": "f27d3f1cd305db5602154a18d85e3ede", "gt2_read.py": "e0d21e012acf77082a26c0e9125a5e5b", "kc_common.py": "45255ae0e361ef787b290d7510554a9d", "kc_g1.py": "a67936253892e757da249c7fbd7023d6", "pack_fleet_ds.py": "bc65dea9510a71ede83f3bffc123c7ea", "tta_bench.py": "176851b1d241b54a283a1e85e2669e30", "ttr_preflight.py": "f8ddcd120ca6ee21ab13fee9d9fba7cc"}
bad = [f for f, h in CODE_MD5.items() if hashlib.md5(open(os.path.join(R, 'fast_build', 'code2', f), 'rb').read()).hexdigest() != h]
assert not bad, 'code md5 mismatch: %s' % bad
UID = re.compile(r'^1\.2\.826\.0\.1\.3680043\.8\.498\.\d+$')
sids = sorted(f[:-4] for f in os.listdir(st))
def ids(p):
    with open(p, newline='') as fh: return [r['StudyInstanceUID'] for r in csv.DictReader(fh)]
fo = set(ids(os.path.join(R, 'rsna_train/folds5.csv'))) | set(ids(os.path.join(R, 'rsna_train/gold74_eval_set.csv')))
assert len(sids) == 4407 and all(UID.match(s) for s in sids) and set(sids) == fo, 'store is not the 4,407 RSNA studies'
for c in ('r7t_labels/real/labels_r7t_b.csv', 'kc/labels/labels_dread_fleet.csv'):
    assert not (set(ids(os.path.join(R, c))) - fo), c
print('RSNA_ONLY_OK store 4407 == folds5 + gold74; code md5s OK', flush=True)

In [ ]:
# 2. Packages pinned to the box env (torch / torchvision of the image kept), exactly as the rental kit did.
import subprocess, sys
pins = [l.strip() for l in open('/kaggle/working/rsna/kit/pins.txt') if l.strip() and not l.lower().startswith(('torch==', 'torchvision=='))] \
    if __import__('os').path.exists('/kaggle/working/rsna/kit/pins.txt') else None
if pins is None:
    import glob; pins = [l.strip() for l in open(glob.glob('/kaggle/input/**/kit/pins.txt', recursive=True)[0]) if l.strip()]
cv = [p for p in pins if p.lower().startswith('opencv')]; rest = [p for p in pins if p not in cv]
import torch, torchvision
open('/tmp/cons.txt', 'w').write('torch==%s\ntorchvision==%s\n' % (torch.__version__.split('+')[0], torchvision.__version__.split('+')[0]))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', '/tmp/cons.txt'] + rest, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps'] + cv, check=True)
print(subprocess.run([sys.executable, '-m', 'pip', 'list', '--format=freeze'], capture_output=True, text=True).stdout[-1500:])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), flush=True)

In [ ]:
# 3. Two members in parallel (one per T4) + a watchdog that stops both at 9.0 h so the commit always ends.
import os, subprocess, time, json, signal, glob, hashlib
R = '/kaggle/working/rsna'; FR = os.path.join(R, 'fleet_runs'); os.makedirs(FR, exist_ok=True)
MEMBERS = [{"key": "s2005", "gpu": 0, "out": "fA_s2005_kc256_dread_friend", "labels": "@R@/kc/labels/labels_dread_fleet.csv", "argv": "--kc --model ttr --ttr_size 256 --backbone convnext_small.fb_in22k_ft_in1k --pretrained_file @R@/r7port/pretrained/convnext_small_in22k_ft_in1k.safetensors --mirror 0 --vbias_prior 1 --aug heavy --train_slices 8 --drop_path 0.3 --dropout 0.3 --ema 0.9995 --epochs 13 --save_epochs 5-13 --bs 1 --accum 8 --lr 1e-4 --head_lr 1e-3 --wd 0.05 --workers 2 --fold -1 --store @R@/kc_store/store --folds @R@/rsna_train/folds5.csv --exclude @R@/rsna_train/gold74_eval_set.csv --gold @R@/rsna_train/gold74_eval_set.csv --grad_ckpt 0 --seed 2005 --labels @R@/kc/labels/labels_dread_fleet.csv --eval_labels @R@/kc/labels/labels_dread_fleet.csv --stop_epoch 12 --out @R@/fleet_runs/fA_s2005_kc256_dread_friend"}, {"key": "s2006", "gpu": 1, "out": "fA_s2006_kc256_r7tb_friend", "labels": "@R@/r7t_labels/real/labels_r7t_b.csv", "argv": "--kc --model ttr --ttr_size 256 --backbone convnext_small.fb_in22k_ft_in1k --pretrained_file @R@/r7port/pretrained/convnext_small_in22k_ft_in1k.safetensors --mirror 0 --vbias_prior 1 --aug heavy --train_slices 8 --drop_path 0.3 --dropout 0.3 --ema 0.9995 --epochs 13 --save_epochs 5-13 --bs 1 --accum 8 --lr 1e-4 --head_lr 1e-3 --wd 0.05 --workers 2 --fold -1 --store @R@/kc_store/store --folds @R@/rsna_train/folds5.csv --exclude @R@/rsna_train/gold74_eval_set.csv --gold @R@/rsna_train/gold74_eval_set.csv --grad_ckpt 0 --seed 2006 --labels @R@/r7t_labels/real/labels_r7t_b.csv --eval_labels @R@/r7t_labels/real/labels_r7t_b.csv --stop_epoch 12 --out @R@/fleet_runs/fA_s2006_kc256_r7tb_friend"}]
WALL = T0 + 9.0 * 3600
procs = {}
for m in MEMBERS:
    argv = m['argv'].replace('@R@', R).split()
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(m['gpu']), HF_HOME=os.path.join(R, 'hf'), HF_HUB_OFFLINE='1',
               PYTHONDONTWRITEBYTECODE='1', OMP_NUM_THREADS='2')
    log = open(os.path.join(FR, m['out'] + '.log'), 'w')
    procs[m['key']] = (subprocess.Popen([sys.executable, 'fleet_train.py'] + argv, cwd=os.path.join(R, 'fast_build', 'code2'),
                                        env=env, stdout=log, stderr=subprocess.STDOUT), log, m)
    print('LAUNCH', m['key'], 'GPU', m['gpu'], ' '.join(argv), flush=True)
stopped = []
while any(p.poll() is None for p, _, _ in procs.values()):
    if time.time() > WALL:
        for k, (p, _, _) in procs.items():
            if p.poll() is None:
                p.send_signal(signal.SIGTERM); stopped.append(k)
        time.sleep(120)
        for p, _, _ in procs.values():
            if p.poll() is None: p.kill()
        break
    time.sleep(60)
rec = {}
for k, (p, log, m) in procs.items():
    log.close()
    d = os.path.join(FR, m['out'])
    files = sorted(glob.glob(os.path.join(d, '**', '*'), recursive=True))
    last = open(os.path.join(FR, m['out'] + '.log'), errors='replace').read().rstrip().splitlines()[-1:]
    rec[k] = dict(rc=p.returncode, stopped_by_watchdog=k in stopped, closing=last[0][:200] if last else '',
                  files={os.path.relpath(f, FR): hashlib.sha256(open(f, 'rb').read()).hexdigest() for f in files if os.path.isfile(f)})
    print(k, 'rc', p.returncode, 'watchdog', k in stopped, 'files', len(rec[k]['files']), flush=True)
rec['hours'] = round((time.time() - T0) / 3600, 2)
json.dump(rec, open(os.path.join(FR, 'friend_receipt.json'), 'w'), indent=1)
# keep only the outputs (drop the code copy and the store links from /kaggle/working)
import shutil
shutil.rmtree(os.path.join(R, 'kc_store'), ignore_errors=True); shutil.rmtree(os.path.join(R, 'fast_build'), ignore_errors=True)
for sub in ('r7port', 'hf', 'kc', 'r7t_labels', 'rsna_train'):
    p = os.path.join(R, sub)
    if os.path.islink(p): os.unlink(p)
print('FRIEND_KC_DONE', json.dumps({k: (v['rc'], v['stopped_by_watchdog']) for k, v in rec.items() if k != 'hours'}), rec['hours'], 'h', flush=True)